# SIR App-Spread Simulation Across Network Topologies

Complex Networks Individual Research Assignment

States:
  S (Susceptible)  - user who could install the app
  I (Infected)     - active user, spreading the app
  R (Recovered)    - bored user, will not install again

Topologies compared:
  - Erdos-Renyi (random)
  - Barabasi-Albert (scale-free)
  - Watts-Strogatz (small-world)
  - Grid / Lattice

In [1]:
import random
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [2]:
# simulation params
N          = 200      # number of nodes
BETA       = 0.12     # per-edge spread probability per step
GAMMA      = 0.05     # recovery (deletion) probability per step
T_MAX      = 150      # maximum simulation steps
N_RUNS     = 30       # independent runs for averaging
INIT_FRAC  = 0.02     # fraction of initially infected nodes

# topology params
ER_P       = 0.05     # Erdos-Renyi connection probability
BA_M       = 3        # Barabasi-Albert edges per new node
WS_K       = 6        # Watts-Strogatz k nearest neighbours
WS_P       = 0.1      # Watts-Strogatz rewiring probability

# colours
COL = {
    "S": "#3B8BD4",   # blue
    "I": "#E8593C",   # coral/red
    "R": "#888780",   # grey
}
TOPO_COLORS = {
    "Erdos-Renyi":      "#534AB7",  # purple
    "Barabasi-Albert":  "#E8593C",  # coral
    "Watts-Strogatz":   "#1D9E75",  # teal
    "Lattice":          "#BA7517",  # amber
}

In [3]:
# 1.  GRAPH BUILDERS

def build_graphs(n: int) -> dict:
    """Return one graph per topology, all with ~N nodes."""
    side = int(np.sqrt(n))
    graphs = {
        "Erdos-Renyi":     nx.erdos_renyi_graph(n, ER_P, seed=SEED),
        "Barabasi-Albert": nx.barabasi_albert_graph(n, BA_M, seed=SEED),
        "Watts-Strogatz":  nx.watts_strogatz_graph(n, WS_K, WS_P, seed=SEED),
        "Lattice":         nx.grid_2d_graph(side, side),
    }
    # Relabel lattice nodes to integers
    graphs["Lattice"] = nx.convert_node_labels_to_integers(graphs["Lattice"])
    return graphs

In [4]:
# 2.  SIR SIMULATION

def sir_step(G: nx.Graph, state: dict, beta: float, gamma: float) -> dict:
    """Advance one time step; return new state dict."""
    new_state = state.copy()
    for node in G.nodes():
        if state[node] == "I":
            # try to recover
            if random.random() < gamma:
                new_state[node] = "R"
            else:
                # try to spread to susceptible neighbours
                for nb in G.neighbors(node):
                    if state[nb] == "S" and random.random() < beta:
                        new_state[nb] = "I"
    return new_state


def run_simulation(G: nx.Graph, beta: float = BETA, gamma: float = GAMMA,
                   t_max: int = T_MAX, init_frac: float = INIT_FRAC,
                   seed: int = None) -> dict:
    """
    Run one SIR simulation on graph G.
    Returns dict with time-series arrays for S, I, R counts.
    """
    if seed is not None:
        random.seed(seed)

    nodes = list(G.nodes())
    n = len(nodes)

    # initialise state
    n_init = max(1, int(n * init_frac))
    state = {v: "S" for v in nodes}
    for v in random.sample(nodes, n_init):
        state[v] = "I"

    S_ts, I_ts, R_ts = [], [], []

    for _ in range(t_max):
        counts = {"S": 0, "I": 0, "R": 0}
        for v in nodes:
            counts[state[v]] += 1
        S_ts.append(counts["S"])
        I_ts.append(counts["I"])
        R_ts.append(counts["R"])

        if counts["I"] == 0:
            # fill remaining steps with last values
            remaining = t_max - len(S_ts)
            S_ts += [counts["S"]] * remaining
            I_ts += [0]           * remaining
            R_ts += [counts["R"]] * remaining
            break

        state = sir_step(G, state, beta, gamma)

    return {
        "S": np.array(S_ts),
        "I": np.array(I_ts),
        "R": np.array(R_ts),
        "extinction_t": len(S_ts) - 1,
        "peak_I":       max(I_ts),
        "peak_t":       int(np.argmax(I_ts)),
        "final_R":      R_ts[-1],
    }


def run_many(G: nx.Graph, n_runs: int = N_RUNS, **kwargs) -> dict:
    """Average SIR curves over multiple independent runs."""
    all_S, all_I, all_R = [], [], []
    peak_Is, final_Rs, peak_ts, ext_ts = [], [], [], []

    for run in range(n_runs):
        res = run_simulation(G, seed=SEED + run, **kwargs)
        all_S.append(res["S"])
        all_I.append(res["I"])
        all_R.append(res["R"])
        peak_Is.append(res["peak_I"])
        final_Rs.append(res["final_R"])
        peak_ts.append(res["peak_t"])
        ext_ts.append(res["extinction_t"])

    t = np.arange(T_MAX)
    return {
        "t":         t,
        "S_mean":    np.mean(all_S, axis=0),
        "I_mean":    np.mean(all_I, axis=0),
        "R_mean":    np.mean(all_R, axis=0),
        "I_std":     np.std(all_I,  axis=0),
        "peak_I":    np.mean(peak_Is),
        "peak_I_std":np.std(peak_Is),
        "final_R":   np.mean(final_Rs),
        "peak_t":    np.mean(peak_ts),
        "ext_t":     np.mean(ext_ts),
    }

In [5]:
# 3.  NETWORK METRICS

def compute_metrics(G: nx.Graph) -> dict:
    """Compute key network-science metrics for the graph."""
    degrees = [d for _, d in G.degree()]
    Gcc = G.subgraph(max(nx.connected_components(G), key=len))

    avg_path = nx.average_shortest_path_length(Gcc) if nx.is_connected(G) else \
               nx.average_shortest_path_length(Gcc)

    return {
        "nodes":        G.number_of_nodes(),
        "edges":        G.number_of_edges(),
        "avg_degree":   np.mean(degrees),
        "max_degree":   max(degrees),
        "clustering":   nx.average_clustering(G),
        "avg_path_len": avg_path,
        "diameter":     nx.diameter(Gcc),
        "density":      nx.density(G),
    }

In [6]:
# 4.  PLOTTING

def plot_network_overview(graphs: dict):
    """Figure 1 – draw each network topology side by side."""
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    fig.suptitle("Network Topologies", fontsize=15, fontweight="bold", y=1.01)

    layouts = {
        "Erdos-Renyi":     lambda g: nx.spring_layout(g, seed=SEED, k=0.4),
        "Barabasi-Albert": lambda g: nx.spring_layout(g, seed=SEED, k=0.3),
        "Watts-Strogatz":  lambda g: nx.circular_layout(g),
        "Lattice":         lambda g: nx.kamada_kawai_layout(g),
    }

    for ax, (name, G) in zip(axes, graphs.items()):
        color = TOPO_COLORS[name]
        pos = layouts[name](G)
        degrees = dict(G.degree())
        node_sizes = [20 + degrees[v] * 8 for v in G.nodes()]

        nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.25, width=0.5,
                               edge_color="#aaaaaa")
        nx.draw_networkx_nodes(G, pos, ax=ax, node_size=node_sizes,
                               node_color=color, alpha=0.85)
        m = compute_metrics(G)
        ax.set_title(
            f"{name}\n"
            f"⟨k⟩={m['avg_degree']:.1f}  C={m['clustering']:.2f}  "
            f"L={m['avg_path_len']:.2f}",
            fontsize=10
        )
        ax.axis("off")

    plt.tight_layout()
    plt.savefig("fig1_topologies.png", dpi=150, bbox_inches="tight")
    print("Saved fig1_topologies.png")
    plt.close()


def plot_degree_distributions(graphs: dict):
    """Figure 2 – degree distributions (log-log for BA, linear for rest)."""
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle("Degree Distributions", fontsize=14, fontweight="bold")

    for ax, (name, G) in zip(axes, graphs.items()):
        color = TOPO_COLORS[name]
        degrees = [d for _, d in G.degree()]
        unique, counts = np.unique(degrees, return_counts=True)
        prob = counts / counts.sum()

        ax.bar(unique, prob, color=color, alpha=0.75, width=0.8)
        ax.set_xlabel("Degree k", fontsize=10)
        ax.set_ylabel("P(k)", fontsize=10)
        ax.set_title(name, fontsize=10, fontweight="bold")

        if name == "Barabasi-Albert":
            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.set_title(name + "\n(log-log scale)", fontsize=10,
                         fontweight="bold")

        ax.grid(True, alpha=0.3, linewidth=0.5)

    plt.tight_layout()
    plt.savefig("fig2_degree_distributions.png", dpi=150, bbox_inches="tight")
    print("Saved fig2_degree_distributions.png")
    plt.close()


def plot_sir_curves(results: dict):
    """Figure 3 – averaged SIR curves, one subplot per topology."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle(
        f"SIR App-Spread Dynamics  (β={BETA}, γ={GAMMA}, R₀={BETA/GAMMA:.1f}, "
        f"N={N}, avg over {N_RUNS} runs)",
        fontsize=13, fontweight="bold"
    )
    axes = axes.flatten()

    for ax, (name, res) in zip(axes, results.items()):
        color = TOPO_COLORS[name]
        t = res["t"]
        n = N

        ax.plot(t, res["S_mean"] / n * 100, color=COL["S"],
                lw=2, label="Susceptible (S)")
        ax.fill_between(t,
                        (res["S_mean"] - res["I_std"]) / n * 100,
                        (res["S_mean"] + res["I_std"]) / n * 100,
                        color=COL["S"], alpha=0.08)

        ax.plot(t, res["I_mean"] / n * 100, color=COL["I"],
                lw=2, label="Active / Infected (I)")
        ax.fill_between(t,
                        (res["I_mean"] - res["I_std"]) / n * 100,
                        (res["I_mean"] + res["I_std"]) / n * 100,
                        color=COL["I"], alpha=0.15)

        ax.plot(t, res["R_mean"] / n * 100, color=COL["R"],
                lw=2, label="Recovered / Bored (R)")

        # annotate peak
        pk = res["peak_I"] / n * 100
        pt = int(res["peak_t"])
        ax.annotate(f"peak {pk:.1f}%",
                    xy=(pt, pk), xytext=(pt + 8, pk + 3),
                    fontsize=8, color=COL["I"],
                    arrowprops=dict(arrowstyle="->", color=COL["I"], lw=0.8))

        ax.set_xlim(0, T_MAX)
        ax.set_ylim(0, 105)
        ax.set_xlabel("Time step", fontsize=10)
        ax.set_ylabel("% of nodes", fontsize=10)
        ax.set_title(name, fontsize=11, fontweight="bold", color=color)
        ax.legend(fontsize=8, loc="upper right")
        ax.grid(True, alpha=0.3, linewidth=0.5)

    plt.tight_layout()
    plt.savefig("fig3_sir_curves.png", dpi=150, bbox_inches="tight")
    print("Saved fig3_sir_curves.png")
    plt.close()


def plot_sir_comparison(results: dict):
    """Figure 4 – overlay I(t) curves for all topologies in one plot."""
    fig, ax = plt.subplots(figsize=(10, 5))

    for name, res in results.items():
        color = TOPO_COLORS[name]
        t = res["t"]
        ax.plot(t, res["I_mean"] / N * 100, color=color, lw=2.5, label=name)
        ax.fill_between(t,
                        (res["I_mean"] - res["I_std"]) / N * 100,
                        (res["I_mean"] + res["I_std"]) / N * 100,
                        color=color, alpha=0.12)

    ax.set_xlabel("Time step", fontsize=12)
    ax.set_ylabel("Active users  I(t)  [% of nodes]", fontsize=12)
    ax.set_title(
        f"Active user prevalence by topology  (β={BETA}, γ={GAMMA}, R₀={BETA/GAMMA:.1f})",
        fontsize=13, fontweight="bold"
    )
    ax.legend(fontsize=11)
    ax.set_xlim(0, T_MAX)
    ax.set_ylim(0)
    ax.grid(True, alpha=0.3, linewidth=0.5)

    plt.tight_layout()
    plt.savefig("fig4_sir_comparison.png", dpi=150, bbox_inches="tight")
    print("Saved fig4_sir_comparison.png")
    plt.close()


def plot_bar_summary(results: dict):
    """Figure 5 – bar charts: peak I, final R, time to peak."""
    metrics = {
        "Peak active users (%)": {n: r["peak_I"] / N * 100 for n, r in results.items()},
        "Final bored users (%)": {n: r["final_R"] / N * 100 for n, r in results.items()},
        "Time to peak (steps)":  {n: r["peak_t"]           for n, r in results.items()},
    }

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle("Simulation Summary Statistics  (averaged over runs)",
                 fontsize=13, fontweight="bold")

    for ax, (label, vals) in zip(axes, metrics.items()):
        names = list(vals.keys())
        values = list(vals.values())
        colors = [TOPO_COLORS[n] for n in names]
        bars = ax.bar(names, values, color=colors, alpha=0.85, width=0.55,
                      edgecolor="white", linewidth=0.8)
        for bar, v in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + max(values) * 0.01,
                    f"{v:.1f}", ha="center", va="bottom", fontsize=10,
                    fontweight="bold")
        ax.set_ylabel(label, fontsize=11)
        ax.set_xticklabels(names, rotation=15, ha="right", fontsize=9)
        ax.set_ylim(0, max(values) * 1.18)
        ax.grid(True, alpha=0.3, axis="y", linewidth=0.5)

    plt.tight_layout()
    plt.savefig("fig5_summary_bars.png", dpi=150, bbox_inches="tight")
    print("Saved fig5_summary_bars.png")
    plt.close()


def plot_r0_sweep(graphs: dict):
    """Figure 6 – final R fraction vs R₀ for each topology."""
    betas  = np.linspace(0.01, 0.30, 18)
    gamma_fixed = 0.05
    fig, ax = plt.subplots(figsize=(10, 5))

    for name, G in graphs.items():
        color = TOPO_COLORS[name]
        final_Rs = []
        for b in betas:
            res = run_many(G, n_runs=15, beta=b, gamma=gamma_fixed,
                           t_max=120, init_frac=INIT_FRAC)
            final_Rs.append(res["final_R"] / N * 100)
        r0s = betas / gamma_fixed
        ax.plot(r0s, final_Rs, color=color, lw=2.5, marker="o",
                markersize=4, label=name)

    ax.axvline(1.0, color="black", lw=1.2, ls="--", alpha=0.6,
               label="Epidemic threshold  R₀=1")
    ax.set_xlabel("Basic reproduction number  R₀ = β/γ", fontsize=12)
    ax.set_ylabel("Final bored users  R(∞)  [%]", fontsize=12)
    ax.set_title("Effect of R₀ on total app adoption across topologies",
                 fontsize=13, fontweight="bold")
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, linewidth=0.5)

    plt.tight_layout()
    plt.savefig("fig6_r0_sweep.png", dpi=150, bbox_inches="tight")
    print("Saved fig6_r0_sweep.png")
    plt.close()


def plot_network_snapshot(graphs: dict, results_raw: dict):
    """Figure 7 – single-run snapshot of node states at t=peak for each topology."""
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    fig.suptitle("Node-state snapshot at peak infection", fontsize=14,
                 fontweight="bold")

    layouts = {
        "Erdos-Renyi":     lambda g: nx.spring_layout(g, seed=SEED, k=0.4),
        "Barabasi-Albert": lambda g: nx.spring_layout(g, seed=SEED, k=0.3),
        "Watts-Strogatz":  lambda g: nx.circular_layout(g),
        "Lattice":         lambda g: nx.kamada_kawai_layout(g),
    }

    state_color_map = {"S": COL["S"], "I": COL["I"], "R": COL["R"]}

    for ax, (name, G) in zip(axes, graphs.items()):
        pos = layouts[name](G)
        # replay simulation to peak step
        nodes_list = list(G.nodes())
        n_init = max(1, int(len(nodes_list) * INIT_FRAC))
        random.seed(SEED)
        state = {v: "S" for v in nodes_list}
        for v in random.sample(nodes_list, n_init):
            state[v] = "I"

        peak_t = int(results_raw[name]["peak_t"])
        for _ in range(peak_t):
            state = sir_step(G, state, BETA, GAMMA)

        node_colors = [state_color_map[state[v]] for v in G.nodes()]
        degrees = dict(G.degree())
        node_sizes = [15 + degrees[v] * 6 for v in G.nodes()]

        nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.2, width=0.4,
                               edge_color="#aaaaaa")
        nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors,
                               node_size=node_sizes, alpha=0.9)
        ax.set_title(f"{name}\n(t = {peak_t})", fontsize=10)
        ax.axis("off")

    legend_elements = [
        mpatches.Patch(color=COL["S"], label="Susceptible (S)"),
        mpatches.Patch(color=COL["I"], label="Active (I)"),
        mpatches.Patch(color=COL["R"], label="Recovered (R)"),
    ]
    fig.legend(handles=legend_elements, loc="lower center", ncol=3,
               fontsize=10, frameon=True, bbox_to_anchor=(0.5, -0.04))

    plt.tight_layout()
    plt.savefig("fig7_snapshot.png", dpi=150, bbox_inches="tight")
    print("Saved fig7_snapshot.png")
    plt.close()


def print_metrics_table(graphs: dict, results: dict):
    """Print a formatted table of network metrics and simulation outcomes."""
    sep = "─" * 90
    print("\n" + sep)
    print(f"{'METRIC':<28}" + "".join(f"{n:>15}" for n in graphs))
    print(sep)

    for name, G in graphs.items():
        pass  # build all metrics
    metric_rows = {n: compute_metrics(G) for n, G in graphs.items()}

    network_rows = [
        ("Nodes",             "nodes",        "{:.0f}"),
        ("Edges",             "edges",        "{:.0f}"),
        ("Avg degree ⟨k⟩",   "avg_degree",   "{:.2f}"),
        ("Max degree",        "max_degree",   "{:.0f}"),
        ("Clustering C",      "clustering",   "{:.3f}"),
        ("Avg path length L", "avg_path_len", "{:.3f}"),
        ("Diameter",          "diameter",     "{:.0f}"),
    ]
    for label, key, fmt in network_rows:
        row = f"{label:<28}"
        for name in graphs:
            val = metric_rows[name][key]
            row += f"{fmt.format(val):>15}"
        print(row)

    print(sep)
    sim_rows = [
        ("Peak active %",     "peak_I",   lambda r: r["peak_I"]  / N * 100, "{:.1f}%"),
        ("Time to peak",      "peak_t",   lambda r: r["peak_t"],             "{:.1f}"),
        ("Final bored %",     "final_R",  lambda r: r["final_R"] / N * 100, "{:.1f}%"),
        ("Extinction step",   "ext_t",    lambda r: r["ext_t"],              "{:.1f}"),
    ]
    for label, key, fn, fmt in sim_rows:
        row = f"{label:<28}"
        for name in graphs:
            val = fn(results[name])
            row += f"{fmt.format(val):>15}"
        print(row)

    print(sep)
    print(f"\nSimulation parameters: β={BETA}, γ={GAMMA}, "
          f"R₀=β/γ={BETA/GAMMA:.2f}, N={N}, runs={N_RUNS}\n")

In [7]:
print("  SIR App-Spread Simulation — Complex Networks Assignment")

# ── build graphs ──────────────────────────────────────────────
print("\n[1/7] Building network topologies ...")
graphs = build_graphs(N)
for name, G in graphs.items():
    m = compute_metrics(G)
    print(f"  {name:<20} nodes={m['nodes']} edges={m['edges']} "
          f"⟨k⟩={m['avg_degree']:.1f} C={m['clustering']:.3f} "
          f"L={m['avg_path_len']:.3f}")

# ── run simulations ───────────────────────────────────────────
print(f"\n[2/7] Running SIR simulations ({N_RUNS} runs × {T_MAX} steps each) ...")
results = {}
for name, G in graphs.items():
    print(f"  Simulating {name} ...", end="", flush=True)
    results[name] = run_many(G, n_runs=N_RUNS)
    print(f"  peak I = {results[name]['peak_I']/N*100:.1f}%  "
          f"final R = {results[name]['final_R']/N*100:.1f}%")

# ── print table ───────────────────────────────────────────────
print_metrics_table(graphs, results)

# ── figures ───────────────────────────────────────────────────
print("[3/7] Plotting topology overview ...")
plot_network_overview(graphs)
print("[4/7] Plotting degree distributions ...")
plot_degree_distributions(graphs)
print("[5/7] Plotting SIR curves ...")
plot_sir_curves(results)
print("[6/7] Plotting comparison overlay ...")
plot_sir_comparison(results)
print("[7/7] Plotting summary bars, R₀ sweep, and snapshots ...")
plot_bar_summary(results)
plot_r0_sweep(graphs)
plot_network_snapshot(graphs, results)

print("\n✓ All figures saved. Simulation complete.")

print("\nGenerated files:")
for f in ["fig1_topologies.png", "fig2_degree_distributions.png",
          "fig3_sir_curves.png", "fig4_sir_comparison.png",
          "fig5_summary_bars.png", "fig6_r0_sweep.png",
          "fig7_snapshot.png"]:
    print(f"  {f}")

  SIR App-Spread Simulation — Complex Networks Assignment

[1/7] Building network topologies ...
  Erdos-Renyi          nodes=200 edges=942 ⟨k⟩=9.4 C=0.046 L=2.591
  Barabasi-Albert      nodes=200 edges=591 ⟨k⟩=5.9 C=0.102 L=2.835
  Watts-Strogatz       nodes=200 edges=600 ⟨k⟩=6.0 C=0.438 L=4.415
  Lattice              nodes=196 edges=364 ⟨k⟩=3.7 C=0.000 L=9.333

[2/7] Running SIR simulations (30 runs × 150 steps each) ...
  Simulating Erdos-Renyi ...  peak I = 79.8%  final R = 100.0%
  Simulating Barabasi-Albert ...  peak I = 69.0%  final R = 98.5%
  Simulating Watts-Strogatz ...  peak I = 59.0%  final R = 99.7%
  Simulating Lattice ...  peak I = 32.6%  final R = 89.7%

──────────────────────────────────────────────────────────────────────────────────────────
METRIC                          Erdos-RenyiBarabasi-Albert Watts-Strogatz        Lattice
──────────────────────────────────────────────────────────────────────────────────────────
Nodes                                   200      